# MiniMax-H3 Image-to-Video (FL2VA) on Google Colab

**Requires:** Colab Pro with A100 (40/80GB) or L4 (24GB). Free T4 will OOM.

Run cells in order. Cell 4 prints your public Cloudflare Tunnel URL. Paste that into the frontend.

In [ ]:
!pip install -q -U git+https://github.com/huggingface/diffusers.git transformers accelerate safetensors sentencepiece protobuf
!pip install -q fastapi uvicorn python-multipart pillow imageio[ffmpeg]

# Download Cloudflare Tunnel binary
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
import torch
from diffusers import ComponentsManager, ModularPipeline

# Enable automatic CPU offloading so the 33B model fits in Colab VRAM
manager = ComponentsManager()
manager.enable_auto_cpu_offload(device="cuda", memory_reserve_margin="6GB")

print("Loading MiniMax-H3 FL2VA pipeline...")
pipe = ModularPipeline.from_pretrained(
    "MiniMaxAI/MiniMax-H3",
    components_manager=manager,
)
pipe.load_components(workflow="fl2va", dtype=torch.bfloat16)
print("Model loaded successfully!")

In [ ]:
%%writefile server.py
import io
import os
import time
import uuid
import base64
import torch
from PIL import Image
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from diffusers.utils.export_utils import encode_video
from diffusers import ComponentsManager, ModularPipeline

app = FastAPI(title="MiniMax-H3 I2V API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

manager = ComponentsManager()
manager.enable_auto_cpu_offload(device="cuda", memory_reserve_margin="6GB")
pipe = ModularPipeline.from_pretrained("MiniMaxAI/MiniMax-H3", components_manager=manager)
pipe.load_components(workflow="fl2va", dtype=torch.bfloat16)

OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def snap_frames(seconds: float, fps: int = 24) -> int:
    """MiniMax-H3 VAE requires frame count to match (17 * n + 5)."""
    frames = max(124, round(seconds * fps))
    while (frames - 5) % 17 != 0:
        frames += 1
    return min(frames, 362)  # Cap at ~15 seconds

@app.get("/health")
def health():
    return {"status": "ok", "model": "MiniMax-H3-FL2VA"}

@app.post("/generate")
async def generate_video(
    image: UploadFile = File(...),
    prompt: str = Form(...),
    duration: float = Form(5.2),
    steps: int = Form(20),
    seed: int = Form(42),
):
    try:
        img_bytes = await image.read()
        init_image = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        num_frames = snap_frames(duration)
        generator = torch.Generator(device="cuda").manual_seed(seed)

        result = pipe(
            image=init_image,
            prompt=prompt,
            num_frames=num_frames,
            num_inference_steps=steps,
            generator=generator,
        )

        video_frames = result.videos[0]
        audio_track = result.audio[0]
        sampling_rate = getattr(result, "audio_sample_rate", 32000)

        out_path = os.path.join(OUTPUT_DIR, f"{uuid.uuid4().hex}.mp4")
        encode_video(
            video_frames,
            fps=24,
            output_path=out_path,
            audio=audio_track,
            audio_sample_rate=sampling_rate,
        )

        with open(out_path, "rb") as f:
            video_b64 = base64.b64encode(f.read()).decode("utf-8")

        return {
            "status": "success",
            "frames": num_frames,
            "video_base64": video_b64,
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
import subprocess
import time
import re

# Start FastAPI in the background
uvicorn_proc = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait briefly for server startup
time.sleep(5)

# Start Cloudflare Quick Tunnel and print the public URL
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in tunnel_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("\n" + "="*60)
        print("YOUR CLOUDFLARE API URL:", match.group(0))
        print("="*60 + "\n")
        break